# YOLOv3 DPU Benchmark
Measures inference speed and power on the KV260 **FPGA DPU (B512)**.

YOLOv3 is an **object detection** model (detects multiple objects + bounding boxes).
Input: 416x416 image. Output: 3 feature maps at different scales.

**Run from Jupyter on the KV260** (`http://192.168.68.60:9090/lab`)

In [ ]:
import os, time, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
N_WARMUP = 5
N_BENCHMARK = 100

# Smart path: check local models/ first, then pynq default
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("dpu_bench.ipynb"))
XMODEL_LOCAL = os.path.join(NOTEBOOK_DIR, "models", "tf_yolov3_voc.xmodel")
XMODEL_PYNQ  = "/root/jupyter_notebooks/pynq-dpu/tf_yolov3_voc.xmodel"
XMODEL = XMODEL_LOCAL if os.path.exists(XMODEL_LOCAL) else XMODEL_PYNQ

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Using xmodel: {XMODEL}")
print(f"Current board power: {read_power_mw()/1000:.2f} W (idle)")

## Step 1 — Load DPU Overlay

In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded!")

## Step 2 — Load YOLOv3 Model
Note the output has **3 tensors** (one per detection scale) unlike ResNet50's single output.

In [ ]:
overlay.load_model(XMODEL)
dpu = overlay.runner

in_t  = dpu.get_input_tensors()
out_t = dpu.get_output_tensors()
print(f"Input shape:   {in_t[0].dims}  (416x416 image)")
print(f"Output shapes: {[t.dims for t in out_t]}")
print(f"  -> 13x13 grid: detects large objects")
print(f"  -> 26x26 grid: detects medium objects")
print(f"  -> 52x52 grid: detects small objects")

## Step 3 — Prepare Buffers

In [ ]:
in_d  = [np.zeros(t.dims, dtype=np.int8) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.int8) for t in out_t]
print("Buffers ready")

## Step 4 — Warmup

In [ ]:
for _ in range(N_WARMUP):
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
print("Warmup done!")

## Step 5 — Benchmark

In [ ]:
power_samples = []
start = time.time()
for i in range(N_BENCHMARK):
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
    if i % 10 == 0:
        power_samples.append(read_power_mw())
elapsed = time.time() - start

avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = N_BENCHMARK / elapsed
latency_ms = elapsed / N_BENCHMARK * 1000
fps_per_watt = fps / avg_power_w

print("=" * 40)
print("YOLOV3 DPU BENCHMARK RESULTS")
print("=" * 40)
print(f"Platform:    KV260 DPU (B512)")
print(f"FPS:         {fps:.1f}")
print(f"Latency:     {latency_ms:.1f} ms/frame")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)

del overlay